# storeguard — Google Colab 학습

무인매장 이상행동(전도/파손/절도 의심) 탐지 모델을 Colab GPU 에서 학습한다.

## 먼저 읽을 것

1. **결제 전에 0단계와 1단계만 먼저 돌려라.** 무료 런타임에서도 GPU 속도를 실측할 수 있다.
   로컬(GTX 1050 Ti)에서 측정된 학습 처리량은 **4.4 clips/s** 다. 이 값보다 몇 배 빠른지 보고
   결제를 판단하면 된다. 추측할 필요가 없다.
2. 런타임 유형을 반드시 **GPU** 로 바꿀 것: 메뉴 `런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU`
3. Colab 세션은 끊긴다. 체크포인트를 **구글 드라이브에 저장**하도록 되어 있어
   끊겨도 `--resume` 으로 이어서 학습할 수 있다.

## 미리 준비할 것 (로컬 PC 에서)

```powershell
cd D:\computervision
$PY = ".\.venv\Scripts\python.exe"
& $PY -m storeguard.data.pack_cache --workers 4        # 24GB -> 5GB (약 7분)
& $PY -m storeguard.tools.make_colab_bundle            # 업로드 묶음 생성
```

그 다음 구글 드라이브의 `MyDrive/storeguard/` 폴더에 아래 두 파일을 올린다.

- `colab_upload/storeguard_code.zip` (수 MB)
- `colab_upload/cache_jpz.tar` (약 5 GB)

## 0단계 — 어떤 GPU 를 받았는지 확인

Colab 은 요청한 GPU 를 항상 주지는 않는다. 실제로 무엇을 받았는지 먼저 본다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
if not torch.cuda.is_available():
    raise SystemExit('GPU 런타임이 아니다. 런타임 > 런타임 유형 변경 > GPU 로 바꿀 것.')
p = torch.cuda.get_device_properties(0)
cc = f'{p.major}.{p.minor}'
tc = {'6': '없음 (Pascal)', '7': '2세대 (Turing/Volta)', '8': '3~4세대 (Ampere/Ada)', '9': '4세대+'}.get(cc[0], '?')
print(f'torch {torch.__version__} | {p.name} | VRAM {p.total_memory/1024**3:.1f} GB | compute {cc} | Tensor Core {tc}')
print('참고: 로컬 GTX 1050 Ti 는 compute 6.1, Tensor Core 없음, 학습 4.4 clips/s')

## 1단계 — 코드만 받아서 속도 먼저 재기 (결제 판단용)

5 GB 캐시를 올리기 전에 **코드만** 가져와 속도를 잰다. 2~3분이면 끝난다.
여기서 나온 clips/s 를 로컬의 4.4 와 비교하면 결제 가치가 바로 보인다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = '/content/drive/MyDrive/storeguard'
assert os.path.isdir(DRIVE), f'{DRIVE} 가 없다. 드라이브에 storeguard 폴더를 만들고 파일을 올릴 것.'
print(os.listdir(DRIVE))

In [ ]:
%%bash
set -e
mkdir -p /content/storeguard
cd /content/storeguard
unzip -oq /content/drive/MyDrive/storeguard/storeguard_code.zip
ls -la
pip install -q pyyaml python-dotenv 2>&1 | tail -1 || true

In [ ]:
# VRAM 과 처리량 실측. 여기가 결제 판단의 핵심이다.
# 로컬(GTX 1050 Ti, batch 8): 학습 4.4 clips/s, 추론 배치1 76.3 ms
%cd /content/storeguard
!mkdir -p reports
!python -m storeguard.tools.vram_probe --config configs/colab.yaml --batches 8,16,32,64,96

### 판단 기준

위 결과의 **학습 clips/s 최대값**을 로컬 4.4 와 비교한다.

| 배수 | 1에폭(16,600클립) 소요 | 6에폭 | 판단 |
|---|---|---|---|
| 1배 (4.4) | 63분 | 6.3시간 | 로컬과 같음. 결제 의미 없음 |
| 5배 (22) | 13분 | 1.3시간 | 확실히 이득 |
| 10배 (44) | 6분 | 40분 | 크게 이득 |
| 20배 (88) | 3분 | 20분 | 반복 실험이 자유로워짐 |

그리고 **OOM 없이 통과한 가장 큰 배치**를 아래 `BATCH` 에 적는다.
가장 큰 배치가 항상 가장 빠르지는 않다. `clips/s` 가 가장 높은 배치를 고르는 것이 맞다.

## 2단계 — 데이터 가져오기

Drive 를 마운트한 채로 파일을 하나씩 읽으면 매우 느리다.
**tar 하나를 로컬 디스크로 복사한 뒤 푸는 것**이 빠르다.

In [ ]:
%%bash
set -e
cd /content/storeguard
echo '--- Drive -> 로컬 디스크 복사 (5GB, 보통 2~5분)'
time cp /content/drive/MyDrive/storeguard/cache_jpz.tar /content/cache_jpz.tar
echo '--- 압축 해제'
time tar -xf /content/cache_jpz.tar -C /content/storeguard
echo '--- 확인'
ls /content/storeguard/data/cache | head -3
ls /content/storeguard/data/cache | wc -l
du -sh /content/storeguard/data/cache
df -h /content | tail -1

## 3단계 — 학습

체크포인트는 **드라이브에 저장**한다. 세션이 끊겨도 살아남는다.

In [ ]:
BATCH  = 32       # 1단계 vram_probe 에서 clips/s 가 가장 높았던 배치로 고칠 것
LR     = 0.0004   # batch 32 기준. 배치를 2배 키우면 lr 도 대략 2배
EPOCHS = 10
NAME   = 'colab_r2p1d'
RUNS   = '/content/drive/MyDrive/storeguard/runs'   # 끊겨도 남는 위치

import shlex
cmd = (f'python -m storeguard.train --config configs/colab.yaml --name {NAME} '
       f'--set train.batch_size={BATCH} --set train.lr={LR} '
       f'--set train.epochs={EPOCHS} --set paths.runs={shlex.quote(RUNS)}')
print(cmd)

In [ ]:
%cd /content/storeguard
!{cmd}

### 세션이 끊겼다면

0~2단계를 다시 실행한 뒤 아래 셀을 돌린다. `last.pt` 부터 이어서 학습한다.

In [ ]:
%cd /content/storeguard
!{cmd} --resume

## 4단계 — 평가

시험 분할은 배포 Validation 세트다. 모델 선택이나 임계값 조정에 쓰지 않는다.

In [ ]:
%cd /content/storeguard
!python -m storeguard.evaluate --run {NAME} --runs-dir {RUNS} --split test
!python -m storeguard.tools.make_report --run {NAME} --runs-dir {RUNS} --reports /content/storeguard/reports

## 5단계 — 결과 가져오기

`runs` 는 이미 드라이브에 있다. 리포트만 옮기면 된다.
로컬 PC 에서는 드라이브의 `runs/colab_r2p1d` 를 `D:\computervision\runs\` 로 내려받아
그대로 대시보드에 쓰면 된다.

```powershell
cd D:\computervision
.\scripts\run_dashboard.cmd --loop      # .env 는 그대로, --run 이름만 맞추면 된다
```

In [ ]:
!cp -r /content/storeguard/reports /content/drive/MyDrive/storeguard/reports_colab
!ls -la /content/drive/MyDrive/storeguard/runs/{NAME}
print('끝. 드라이브의 runs/', NAME, ' 를 로컬 runs\\ 폴더로 내려받아 쓰면 된다.')

## 주의할 점

- **프레임 캐시는 손실 압축(JPEG 품질 95)이다.** 로컬 `.npy` 와 비교했을 때
  예측 클래스 일치율 99.58 %, 점수 차이 평균 0.0098 로 측정되었다. 학습과 평가를 같은
  형식으로 하면 문제없다. 로컬 `.npy` 로 학습한 가중치와 직접 비교할 때만 이 차이를 감안한다.
- **Colab 표준 런타임은 vCPU 가 2개다.** GPU 가 빨라지면 데이터 로더가 병목이 될 수 있다.
  학습 로그의 실제 clips/s 가 vram_probe 값보다 한참 낮으면 `train.num_workers` 를 3~4 로 올린다.
- **배치를 키우면 lr 도 같이 올린다.** 배치만 키우고 lr 을 그대로 두면 학습이 덜 진행된다.
- **test 분할은 한 번만 쓴다.** 임계값 조정은 val 로만 한다.